In [ ]:
# ══════════════════════════════════════════════
# PRELUDE — run this cell first
# Added by fix pass. Everything below your own code is unchanged.
# ══════════════════════════════════════════════

import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from datetime import datetime, timezone

class Stale(RuntimeError): pass
class Unconverged(RuntimeError): pass
class TooFewObs(RuntimeError): pass


def safe_at(obj, i=-1, col=None):
    """float() on a 1-element Series is deprecated. Handles yfinance MultiIndex columns."""
    s = obj[col] if col is not None else obj
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    a = np.asarray(s.dropna()).ravel()
    if a.size == 0:
        raise Stale("empty series")
    return float(a[i])


def safe_last(obj, col=None):
    return safe_at(obj, -1, col)


def unmute_convergence():
    """Let convergence failures through. They were being swallowed by filterwarnings('ignore')."""
    try:
        import statsmodels.tools.sm_exceptions as _s
        for _n in ("ConvergenceWarning", "EstimationWarning", "ValueWarning"):
            if hasattr(_s, _n):
                warnings.filterwarnings("always", category=getattr(_s, _n))
    except Exception:
        pass
    try:
        from arch.utility.exceptions import DataScaleWarning
        warnings.filterwarnings("always", category=DataScaleWarning)
    except Exception:
        pass


# ── CONTRACT: pin which gold series this notebook uses ──────────────
#   'front_intraday' = GC=F 30-min  (the contract you trade — DEFAULT)
#   'front_daily'    = GC=F daily   (often prints spot, not the future)
#   'spot'           = XAUUSD
LIVE_CONTRACT = "front_intraday"


def get_ctx(contract=LIVE_CONTRACT):
    if contract == "front_intraday":
        gc_df = yf.download("GC=F", period="5d", interval="30m", progress=False)
    elif contract == "front_daily":
        gc_df = yf.download("GC=F", period="1mo", interval="1d", progress=False)
    elif contract == "spot":
        gc_df = yf.download("XAUUSD=X", period="5d", interval="30m", progress=False)
    else:
        raise ValueError(contract)

    gc = safe_last(gc_df, "Close")
    gld = safe_last(yf.download("GLD", period="5d", progress=False), "Close")

    def _s(t, d):
        try:
            return safe_last(yf.download(t, period="5d", progress=False), "Close")
        except Exception:
            return d

    if not 1000 < gc < 12000:
        raise Stale(f"GC={gc:,.2f} implausible — bad fetch")
    if not 100 < gld < 1200:
        raise Stale(f"GLD={gld:,.2f} implausible — bad fetch")
    ratio = gc / gld
    if not 9.5 <= ratio <= 12.5:
        raise Stale(f"GLD->gold ratio {ratio:.3f}x outside [9.5, 12.5] — "
                    f"prices are from different dates")

    # known failure point: GC=F daily and 30m disagree
    try:
        _d = safe_last(yf.download("GC=F", period="1mo", interval="1d", progress=False), "Close")
        _i = safe_last(yf.download("GC=F", period="5d", interval="30m", progress=False), "Close")
        if abs(_d - _i) / _i > 0.005:
            print(f"  !! GC=F daily {_d:,.2f} vs 30m {_i:,.2f} — ${abs(_d-_i):,.1f} apart.")
            print(f"     Using '{contract}'. VERIFY THE SETTLE IN QUANTOWER.")
    except Exception:
        pass

    ctx = dict(gc=gc, gld=gld, ratio=ratio, contract=contract,
               asof=str(gc_df.index[-1]),
               vix=_s("^VIX", np.nan), gvz=_s("^GVZ", np.nan), rf=_s("^IRX", 4.0) / 100)
    print(f"  contract : {contract}")
    print(f"  GC {gc:>10,.2f}   GLD {gld:>8,.2f}   ratio {ratio:.4f}x   <-- NOT 10.0")
    print(f"  VIX {ctx['vix']:.2f}   GVZ {ctx['gvz']:.1f}   rf {ctx['rf']:.2%}   as of {ctx['asof']}")
    return ctx


def need_obs(n, floor, label=""):
    if n < floor:
        raise TooFewObs(f"{label}: {n} observations, need >= {floor}. Do not report this fit.")


def need_fresh(as_of, days=7, label="field"):
    age = (datetime.now(timezone.utc).date() - datetime.fromisoformat(as_of).date()).days
    if age > days:
        raise Stale(f"{label} written {as_of} ({age}d ago) — EXPIRED. Rewrite or delete it.")
    return True


def need_converged(res, states=None, label="model"):
    conv = getattr(res, "converged", None)
    if conv is None and isinstance(getattr(res, "mle_retvals", None), dict):
        conv = res.mle_retvals.get("converged", True)
    if conv is False:
        raise Unconverged(f"{label}: optimiser did not converge. Not a regime classification.")
    if states is not None:
        u, c = np.unique(np.asarray(states), return_counts=True)
        if len(u) < 2:
            raise Unconverged(f"{label}: {len(u)} distinct state — a flat line, not regimes.")
        if c.min() / c.sum() < 0.05:
            raise Unconverged(f"{label}: minority state is {c.min()/c.sum():.1%} — "
                              f"outlier detector, not a regime model.")


def kelly_cap(f, frac=0.25, cap=0.20):
    out = float(np.clip(f * frac, -cap, cap))
    if abs(f) > 1:
        print(f"  ! raw f*={f:.2f} implies {f*100:.0f}% of capital (small-sample artefact). "
              f"Using {out:.1%}.")
    return out


LIVE_CTX   = get_ctx()
LIVE_RATIO = LIVE_CTX["ratio"]   # use instead of 10
LIVE_GC    = LIVE_CTX["gc"]

# NOTE: named LIVE_* on purpose — GOLD is your hex colour in 15 cells,
#       and RATIO / CONTRACT are already used in cells 44-45.


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# LIVE GOLD OPTIONS CHAIN  —  replaces the synthetic OI generator
#
# WAS:  oi_base = 7000*np.exp(-9*distance)
#       call_oi = oi_base*(1+0.12*np.sin(K/50))
#       put_oi  = oi_base*(1+0.12*np.cos(K/50))
#   -> a Gaussian bump with a sine wave on it. Every level derived from it
#      (GEX flip, gamma wall, call/put wall, max pain) was a property of the
#      trig function, not of dealer positioning. The sin/cos quarter-cycle
#      offset is why call wall and put wall always came out ~78 apart.
#
# NOW: real GLD open interest and real implied vol, strikes translated to
#      gold with the live ratio.
# ══════════════════════════════════════════════════════════════════════

import yfinance as yf


def fetch_gold_chain(max_dte=60, strike_window=0.15, verbose=True):
    """
    Real GLD chain -> gold-denominated DataFrame.

    Columns match what the rest of this notebook expects:
        expiry (dte, int), strike (GOLD $), moneyness,
        iv_call, iv_put, call_oi, put_oi, call_vol, put_vol
    Greeks are added afterwards by the notebook's own bs_greeks().
    """
    tk = yf.Ticker("GLD")
    exps = tk.options
    if not exps:
        raise RuntimeError("no GLD expirations returned — market closed or fetch failed")

    today = pd.Timestamp.now().normalize()
    rows = []
    used = []

    for e in exps:
        dte = (pd.Timestamp(e) - today).days
        if dte <= 0 or dte > max_dte:
            continue
        try:
            oc = tk.option_chain(e)
        except Exception:
            continue

        c = oc.calls[["strike", "openInterest", "impliedVolatility", "volume"]].copy()
        p = oc.puts[["strike", "openInterest", "impliedVolatility", "volume"]].copy()
        c.columns = ["strike", "call_oi", "iv_call", "call_vol"]
        p.columns = ["strike", "put_oi", "iv_put", "put_vol"]

        m = c.merge(p, on="strike", how="outer")
        m["expiry"] = dte
        rows.append(m)
        used.append((e, dte))

    if not rows:
        raise RuntimeError(f"no expirations within {max_dte} DTE")

    df = pd.concat(rows, ignore_index=True)

    # GLD strikes -> gold. LIVE_RATIO comes from the PRELUDE cell.
    df["strike"] = df["strike"] * LIVE_RATIO
    df["moneyness"] = df["strike"] / LIVE_GC

    df[["call_oi", "put_oi", "call_vol", "put_vol"]] = \
        df[["call_oi", "put_oi", "call_vol", "put_vol"]].fillna(0)

    # yfinance returns 0.0 or absurd IVs on illiquid strikes; interpolate
    for col in ("iv_call", "iv_put"):
        bad = (df[col] <= 0.01) | (df[col] > 5.0) | df[col].isna()
        df.loc[bad, col] = np.nan
        df[col] = df.groupby("expiry")[col].transform(
            lambda s: s.interpolate(limit_direction="both"))
        df[col] = df[col].fillna(df[col].median())

    lo, hi = LIVE_GC * (1 - strike_window), LIVE_GC * (1 + strike_window)
    df = df[(df.strike >= lo) & (df.strike <= hi)]
    df = df[(df.call_oi + df.put_oi) > 0].reset_index(drop=True)

    if len(df) < 20:
        raise RuntimeError(f"only {len(df)} usable strikes after filtering — chain too thin")

    if verbose:
        print(f"  LIVE CHAIN  |  GLD -> gold @ {LIVE_RATIO:.4f}x")
        print(f"    expiries : {', '.join(f'{e}({d}d)' for e, d in used)}")
        print(f"    strikes  : {len(df)} rows, ${df.strike.min():,.0f}-${df.strike.max():,.0f}")
        print(f"    total OI : {df.call_oi.sum():,.0f} calls / {df.put_oi.sum():,.0f} puts")
        print(f"    ATM IV   : {df.iloc[(df.strike - LIVE_GC).abs().argmin()].iv_call:.1%}")
    return df


def true_max_pain(df, max_dte=14):
    """
    Max pain = strike that MINIMISES total payout to option HOLDERS.

    WAS: near_K.total_oi.idxmin() -- the strike with the LEAST open interest,
    which is always a far illiquid strike. That is what produced max pain of
    $3,723 while gold traded $4,380.
    """
    near = df[df.expiry <= max_dte]
    if near.empty:
        near = df[df.expiry == df.expiry.min()]
    c = near.groupby("strike").call_oi.sum()
    p = near.groupby("strike").put_oi.sum()
    ks = np.sort(near.strike.unique())
    pain = [((np.maximum(S - c.index.values, 0) * c.values).sum() +
             (np.maximum(p.index.values - S, 0) * p.values).sum()) for S in ks]
    return float(ks[int(np.argmin(pain))])


def true_gex_flip(df, spot, grid_pct=0.25, n=401):
    """
    Flip = the SPOT at which net dealer gamma crosses zero, found by
    re-evaluating gamma across a grid and taking the crossing NEAREST spot.

    WAS: sorted_K.iloc[crossings[0]]['strike'] -- the FIRST crossing in
    strike order, i.e. the lowest strike. Same bug class as the $205 flip.
    Returns None if no crossing exists in range (an honest answer).
    """
    grid = np.linspace(spot * (1 - grid_pct), spot * (1 + grid_pct), n)
    curve = []
    for S in grid:
        g = 0.0
        for _, r in df.iterrows():
            t = max(r.expiry / 365, 1 / 8760)
            gc_ = bs_greeks(S, r.strike, t, RISK_FREE, r.iv_call, 'call')['gamma']
            gp_ = bs_greeks(S, r.strike, t, RISK_FREE, r.iv_put, 'put')['gamma']
            g += (gc_ * r.call_oi - gp_ * r.put_oi) * 100 * S
        curve.append(g)
    curve = np.array(curve)
    cross = np.where(np.diff(np.sign(curve)) != 0)[0]
    if not len(cross):
        return None, pd.DataFrame({"spot": grid, "net_gex": curve})
    cands = [grid[i] - curve[i] * (grid[i + 1] - grid[i]) / (curve[i + 1] - curve[i])
             for i in cross if curve[i + 1] != curve[i]]
    flip = float(min(cands, key=lambda x: abs(x - spot))) if cands else None
    return flip, pd.DataFrame({"spot": grid, "net_gex": curve})


def side_walls(by_K, spot):
    """
    Call wall must be ABOVE spot, put wall BELOW.

    WAS: by_K.call_oi.idxmax() / by_K.put_oi.idxmax() over ALL strikes with no
    side restriction. A call wall below spot is not resistance. That is how the
    put wall ($4,383) ended up above the gamma wall ($4,363), sitting on spot.
    """
    above = by_K[by_K.strike > spot]
    below = by_K[by_K.strike < spot]
    cw = float(above.loc[above.call_oi.idxmax(), "strike"]) if len(above) else np.nan
    pw = float(below.loc[below.put_oi.idxmax(), "strike"]) if len(below) else np.nan
    return cw, pw


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from scipy.stats import norm
from datetime import datetime
import yfinance as yf

import warnings
warnings.filterwarnings('ignore'); unmute_convergence()


#══════════════════════════════════════════════
# CONFIG
#══════════════════════════════════════════════

DAYS_TO_EXP = 30
T = DAYS_TO_EXP/365

GC_MULT = 100

ACCOUNT = 50000
RISK_PCT = 0.01

BG = '#0d0d0d'
GOLD = '#FFD700'
RED = '#FF4444'
GRN = '#00FF88'
BLUE = '#4488FF'
MUTE = '#555555'


#══════════════════════════════════════════════
# LIVE MARKET DATA
#══════════════════════════════════════════════

def get_gold_spot():

    symbols = ["GC=F", "GLD"]

    for sym in symbols:

        try:

            data = yf.download(
                sym,
                period="5d",
                progress=False,
                auto_adjust=True
            )

            if len(data):

                return float(
                    data["Close"].dropna().iloc[-1]
                )

        except:
            pass

    raise ValueError("Unable to retrieve gold price.")


def get_gold_iv():

    try:

        data = yf.download(
            "^GVZ",
            period="5d",
            progress=False,
            auto_adjust=True
        )

        if len(data):

            return float(
                data["Close"].dropna().iloc[-1]
            )/100

    except:
        pass

    # fallback
    return 0.18


def get_risk_free():

    try:

        data = yf.download(
            "^IRX",
            period="5d",
            progress=False,
            auto_adjust=True
        )

        if len(data):

            return float(
                data["Close"].dropna().iloc[-1]
            )/100

    except:
        pass

    # fallback
    return 0.045


GOLD_SPOT = get_gold_spot()
ATM_IV = get_gold_iv()
RISK_FREE = get_risk_free()

today = datetime.now().strftime("%Y-%m-%d")

print()
print("Date:", today)
print("Gold Spot:", round(GOLD_SPOT,2))
print("ATM IV:", round(ATM_IV*100,2), "%")
print("Risk Free:", round(RISK_FREE*100,2), "%")


#══════════════════════════════════════════════
# BLACK-SCHOLES GREEKS
#══════════════════════════════════════════════

def bs_greeks(S, K, T, r, sigma, flag='call'):

    if T <= 1e-6 or sigma <= 1e-6:

        intrinsic = max(S-K,0) if flag=='call' else max(K-S,0)

        return dict(
            price=intrinsic,
            delta=1 if S>K else 0,
            gamma=0,
            vega=0,
            theta=0,
            vanna=0,
            charm=0,
            volga=0
        )

    d1 = (
        np.log(S/K)
        + (r+0.5*sigma**2)*T
    )/(sigma*np.sqrt(T))

    d2 = d1 - sigma*np.sqrt(T)

    phi = norm.pdf(d1)

    Nd1 = norm.cdf(d1) if flag=='call' else norm.cdf(-d1)
    Nd2 = norm.cdf(d2) if flag=='call' else norm.cdf(-d2)

    sign = 1 if flag=='call' else -1

    price = sign*(S*Nd1 - K*np.exp(-r*T)*Nd2)

    delta = sign*Nd1

    gamma = phi/(S*sigma*np.sqrt(T))

    vega = S*phi*np.sqrt(T)/100

    theta = (
        -(S*phi*sigma)/(2*np.sqrt(T))
        - sign*r*K*np.exp(-r*T)*(norm.cdf(sign*d2))
    )/365

    vanna = -phi*d2/sigma

    charm = (
        -phi*(2*r*T - d2*sigma*np.sqrt(T))
        /(2*T*sigma*np.sqrt(T))
    )

    volga = vega*(d1*d2/sigma)

    return dict(
        price=price,
        delta=delta,
        gamma=gamma,
        vega=vega,
        theta=theta,
        vanna=vanna,
        charm=charm,
        volga=volga
    )


#══════════════════════════════════════════════
# STRIKES
#══════════════════════════════════════════════

# ── OPTION CHAIN — LIVE (was synthetic sin/cos OI) ────────────────
expiries = [7,14,21,30,45,60]   # kept for reference; real DTEs come from the chain

_live = fetch_gold_chain(max_dte=60)
strikes = np.sort(_live.strike.unique())

chain = []

for _, _r in _live.iterrows():

    exp = int(_r.expiry)
    t = exp/365
    K = float(_r.strike)
    m = K/GOLD_SPOT

    iv_call = float(_r.iv_call)
    iv_put = float(_r.iv_put)

    call_oi = int(_r.call_oi)
    put_oi = int(_r.put_oi)

    cg = bs_greeks(
        GOLD_SPOT,
        K,
        t,
        RISK_FREE,
        iv_call,
        'call'
    )

    pg = bs_greeks(
        GOLD_SPOT,
        K,
        t,
        RISK_FREE,
        iv_put,
        'put'
    )

    chain.append(

        dict(

            expiry=exp,

            strike=K,

            moneyness=m,

            iv_call=iv_call,
            iv_put=iv_put,

            call_oi=call_oi,
            put_oi=put_oi,

            call_price=cg['price'],
            put_price=pg['price'],

            call_delta=cg['delta'],
            put_delta=pg['delta'],

            call_gamma=cg['gamma'],
            put_gamma=pg['gamma'],

            call_vega=cg['vega'],
            put_vega=pg['vega'],

            call_theta=cg['theta'],
            put_theta=pg['theta'],

            call_vanna=cg['vanna'],
            put_vanna=pg['vanna'],

            call_charm=cg['charm'],
            put_charm=pg['charm'],

            call_volga=cg['volga'],
            put_volga=pg['volga']

        )

    )


df = pd.DataFrame(chain)
#══════════════════════════════════════════════
# AGGREGATE BY STRIKE
#══════════════════════════════════════════════

by_K = df.groupby('strike').apply(

    lambda x: pd.Series({

        'net_gex':
        (
            (
                x.call_gamma*x.call_oi
                -
                x.put_gamma*x.put_oi
            )*100*GOLD_SPOT
        ).sum(),

        'net_vanna':
        (
            (
                x.call_vanna*x.call_oi
                -
                x.put_vanna*x.put_oi
            )*100
        ).sum(),

        'net_charm':
        (
            (
                x.call_charm*x.call_oi
                -
                x.put_charm*x.put_oi
            )*100
        ).sum(),

        'net_vega':
        (
            (
                x.call_vega*x.call_oi
                +
                x.put_vega*x.put_oi
            )*100
        ).sum(),

        'call_oi':
        x.call_oi.sum(),

        'put_oi':
        x.put_oi.sum()

    })

).reset_index()


#══════════════════════════════════════════════
# KEY LEVELS
#══════════════════════════════════════════════

sorted_K = by_K.sort_values('strike')

crossings = np.where(
    np.diff(
        np.sign(
            sorted_K.net_gex.values
        )
    ) != 0
)[0]


# ── KEY LEVELS — fixed selection rules ────────────────────────────
# flip: crossing NEAREST spot on a spot grid (was: crossings[0] = lowest strike)
gex_flip, _gex_curve = true_gex_flip(_live, GOLD_SPOT)
if gex_flip is None:
    gex_flip = GOLD_SPOT*0.92
    print("  note: no gamma crossing within +/-25% of spot — using fallback")

gamma_wall = float(
    by_K.loc[
        by_K.net_gex.abs().idxmax(),
        'strike'
    ]
)

# walls restricted to the correct side of spot (was: idxmax over ALL strikes)
call_wall, put_wall = side_walls(by_K, GOLD_SPOT)


#══════════════════════════════════════════════
# MAX PAIN
#══════════════════════════════════════════════

# strike minimising total payout to option HOLDERS
# (was: total_oi.idxmin() = the LEAST-traded strike, always a far extreme)
max_pain = true_max_pain(_live, max_dte=14)


#══════════════════════════════════════════════
# PRINT LEVELS
#══════════════════════════════════════════════

print()

print("Gold Spot      :", round(GOLD_SPOT,2))

print("GEX Flip       :", round(gex_flip,2))

print("Gamma Wall     :", round(gamma_wall,2))

print("Call Wall      :", round(call_wall,2))

print("Put Wall       :", round(put_wall,2))

print("Max Pain       :", round(max_pain,2))


#══════════════════════════════════════════════
# ATM CALL
#══════════════════════════════════════════════

K_call = round(GOLD_SPOT/10)*10

g0 = bs_greeks(

    GOLD_SPOT,

    K_call,

    T,

    RISK_FREE,

    ATM_IV,

    'call'

)

premium = g0['price']


#══════════════════════════════════════════════
# SCENARIOS
#══════════════════════════════════════════════

scenarios = [

    -0.10,

    -0.05,

    0.0,

    0.05,

    0.10,

    0.15

]


probs = [

    0.10,

    0.15,

    0.30,

    0.25,

    0.12,

    0.08

]


#══════════════════════════════════════════════
# EV TABLE
#══════════════════════════════════════════════

print()

print("="*80)

print("GOLD EXPECTED VALUE TABLE")

print("="*80)

print()

print(
f"ATM Strike = ${K_call:,.0f}"
)

print(
f"Premium = ${premium:.2f}/oz"
)

print(
f"Contract Cost = ${premium*GC_MULT:,.0f}"
)

print()

print(
f"{'Scenario':>10}"
f"{'Spot':>10}"
f"{'PnL':>12}"
f"{'Prob':>10}"
f"{'EV':>12}"
)

print("-"*80)

total_ev = 0

pnls = []

for chg,prob in zip(

        scenarios,

        probs

):

    S_new = GOLD_SPOT*(1+chg)

    iv_new = ATM_IV*(1-0.3*chg)

    iv_new = max(iv_new,0.05)

    g_new = bs_greeks(

        S_new,

        K_call,

        1/365,

        RISK_FREE,

        iv_new,

        'call'

    )

    total_pnl = (

        g_new['price']

        -

        premium

    )*GC_MULT

    pnls.append(total_pnl)

    ev = total_pnl*prob

    total_ev += ev

    print(

        f"{chg:+.0%}"

        f"{S_new:>10,.0f}"

        f"{total_pnl:>12,.0f}"

        f"{prob:>10.0%}"

        f"{ev:>12,.0f}"

    )


print("-"*80)

print()

print(

"EXPECTED VALUE =",

round(total_ev,0)

)

if total_ev > 0:

    print("POSITIVE EV ✅")

else:

    print("NEGATIVE EV ❌")

#══════════════════════════════════════════════
# FIGURE 1 — GREEKS DASHBOARD
#══════════════════════════════════════════════

fig = plt.figure(

    figsize=(20,14),

    facecolor=BG

)

fig.suptitle(

f'GOLD OPTIONS DASHBOARD\n{today}\nSpot ${GOLD_SPOT:,.0f}',

color=GOLD,

fontsize=14,

fontweight='bold',

y=0.98

)

gs = gridspec.GridSpec(

    3,

    3,

    figure=fig,

    hspace=0.45,

    wspace=0.35

)

ax1 = fig.add_subplot(gs[0,0])
ax2 = fig.add_subplot(gs[0,1])
ax3 = fig.add_subplot(gs[0,2])

ax4 = fig.add_subplot(gs[1,0])
ax5 = fig.add_subplot(gs[1,1])
ax6 = fig.add_subplot(gs[1,2])

ax7 = fig.add_subplot(gs[2,0])
ax8 = fig.add_subplot(gs[2,1])
ax9 = fig.add_subplot(gs[2,2])


for ax in [

    ax1,ax2,ax3,

    ax4,ax5,ax6,

    ax7,ax8,ax9

]:

    ax.set_facecolor('#111111')

    for sp in ax.spines.values():

        sp.set_color('#333')

    ax.tick_params(

        colors='#888',

        labelsize=8

    )


ks = by_K.strike.values

gex = by_K.net_gex.values

van = by_K.net_vanna.values

cha = by_K.net_charm.values


#════════════════════════════════════
# NET GEX
#════════════════════════════════════

ax1.bar(

    ks,

    gex,

    width=15,

    color=[GRN if x>=0 else RED for x in gex],

    alpha=0.85

)

ax1.axhline(

    0,

    color=MUTE,

    lw=0.5

)

ax1.axvline(

    GOLD_SPOT,

    color=GOLD,

    lw=1.2

)

ax1.axvline(

    gamma_wall,

    color=BLUE,

    lw=1,

    ls='--'

)

ax1.axvline(

    gex_flip,

    color=RED,

    lw=1,

    ls='--'

)

ax1.set_title(

    'Net GEX by Strike',

    color='white'

)

ax1.ticklabel_format(

    style='sci',

    axis='y',

    scilimits=(0,0)

)


#════════════════════════════════════
# VANNA
#════════════════════════════════════

ax2.bar(

    ks,

    van,

    width=15,

    color=['#00CCBB' if x>=0 else '#FFA500' for x in van],

    alpha=0.85

)

ax2.axhline(

    0,

    color=MUTE,

    lw=0.5

)

ax2.axvline(

    GOLD_SPOT,

    color=GOLD

)

ax2.set_title(

    'Net Vanna',

    color='white'

)

ax2.ticklabel_format(

    style='sci',

    axis='y',

    scilimits=(0,0)

)


#════════════════════════════════════
# CHARM
#════════════════════════════════════

ax3.bar(

    ks,

    cha,

    width=15,

    color=['#AA66FF' if x>=0 else '#888888' for x in cha],

    alpha=0.85

)

ax3.axhline(

    0,

    color=MUTE,

    lw=0.5

)

ax3.axvline(

    GOLD_SPOT,

    color=GOLD

)

ax3.set_title(

    'Net Charm',

    color='white'

)

ax3.ticklabel_format(

    style='sci',

    axis='y',

    scilimits=(0,0)

)


#════════════════════════════════════
# VOL SMILE
#════════════════════════════════════

smile_df = df[df.expiry==30]

ax4.plot(

    smile_df.strike,

    smile_df.iv_call*100,

    color=GRN,

    lw=1.5,

    label='Call IV'

)

ax4.plot(

    smile_df.strike,

    smile_df.iv_put*100,

    color=RED,

    lw=1.5,

    label='Put IV'

)

ax4.axvline(

    GOLD_SPOT,

    color=GOLD,

    ls='--'

)

ax4.legend(

    fontsize=7

)

ax4.set_title(

    'Vol Smile',

    color='white'

)


#════════════════════════════════════
# DELTA
#════════════════════════════════════

d30 = df[df.expiry==30]

ax5.plot(

    d30.strike,

    d30.call_delta,

    color=GRN,

    lw=1.5,

    label='Call Δ'

)

ax5.plot(

    d30.strike,

    d30.put_delta,

    color=RED,

    lw=1.5,

    label='Put Δ'

)

ax5.axvline(

    GOLD_SPOT,

    color=GOLD,

    ls='--'

)

ax5.legend(

    fontsize=7

)

ax5.set_title(

    'Delta',

    color='white'

)


#════════════════════════════════════
# GAMMA
#════════════════════════════════════

ax6.plot(

    d30.strike,

    d30.call_gamma*1e4,

    color=BLUE,

    lw=1.5

)

ax6.axvline(

    GOLD_SPOT,

    color=GOLD,

    ls='--'

)

ax6.set_title(

    'Gamma ×10⁴',

    color='white'

)


#════════════════════════════════════
# OPEN INTEREST
#════════════════════════════════════

ax7.bar(

    by_K.strike,

    by_K.call_oi/1000,

    width=12,

    color=GRN,

    alpha=0.7

)

ax7.bar(

    by_K.strike,

    -by_K.put_oi/1000,

    width=12,

    color=RED,

    alpha=0.7

)

ax7.axhline(

    0,

    color=MUTE

)

ax7.axvline(

    GOLD_SPOT,

    color=GOLD

)

ax7.set_title(

    'Call / Put OI',

    color='white'

)


#════════════════════════════════════
# PNL SCENARIOS
#════════════════════════════════════

bars = ax8.bar(

    [f'{x:+.0%}' for x in scenarios],

    pnls,

    color=[GRN if x>=0 else RED for x in pnls],

    alpha=0.85

)

ax8.axhline(

    0,

    color=MUTE

)

ax8.set_title(

    'ATM Call PnL',

    color='white'

)


#════════════════════════════════════
# SCORECARD
#════════════════════════════════════

ax9.axis('off')

items = [

    ('Spot',f'${GOLD_SPOT:,.0f}','white'),

    ('GEX Flip',f'${gex_flip:,.0f}',RED),

    ('Gamma Wall',f'${gamma_wall:,.0f}',BLUE),

    ('Call Wall',f'${call_wall:,.0f}',GRN),

    ('Put Wall',f'${put_wall:,.0f}',RED),

    ('Max Pain',f'${max_pain:,.0f}',GOLD),

    ('ATM IV',f'{ATM_IV*100:.1f}%','white'),

    ('EV',f'${total_ev:+,.0f}',GRN if total_ev>0 else RED)

]

y = 0.95

for label,val,col in items:

    ax9.text(

        0.02,

        y,

        label,

        transform=ax9.transAxes,

        color='#888',

        fontsize=8.5

    )

    ax9.text(

        0.98,

        y,

        val,

        transform=ax9.transAxes,

        ha='right',

        color=col,

        fontsize=8.5,

        fontweight='bold'

    )

    y -= 0.10


plt.savefig(

    f'gold_greeks_2d_{today}.png',

    dpi=130,

    bbox_inches='tight',

    facecolor=BG

)

plt.show()

print()

print("✅ Figure 1 saved")

#══════════════════════════════════════════════
# FIGURE 2 — 3D SURFACES
#══════════════════════════════════════════════

from mpl_toolkits.mplot3d import Axes3D

spot_range = np.linspace(
    GOLD_SPOT*0.85,
    GOLD_SPOT*1.15,
    60
)

iv_range = np.linspace(
    0.10,
    0.30,
    50
)

S_g, IV_g = np.meshgrid(
    spot_range,
    iv_range
)

K_atm = K_call

t30 = 30/365


DELTA_S = np.zeros_like(S_g)
GAMMA_S = np.zeros_like(S_g)
VANNA_S = np.zeros_like(S_g)
VOLGA_S = np.zeros_like(S_g)
PNL_S = np.zeros_like(S_g)


for i in range(S_g.shape[0]):

    for j in range(S_g.shape[1]):

        g = bs_greeks(

            S_g[i,j],

            K_atm,

            t30,

            RISK_FREE,

            IV_g[i,j],

            'call'

        )

        DELTA_S[i,j] = g['delta']

        GAMMA_S[i,j] = g['gamma']*1e4

        VANNA_S[i,j] = g['vanna']

        VOLGA_S[i,j] = g['volga']

        PNL_S[i,j] = (

            g['price']

            -

            premium

        )*GC_MULT


fig3d = plt.figure(

    figsize=(22,16),

    facecolor=BG

)

fig3d.suptitle(

    f'GOLD OPTIONS 3D SURFACES\n{today}',

    color=GOLD,

    fontsize=14,

    fontweight='bold'

)


surfaces = [

    (231, DELTA_S, 'Delta Surface', 'RdYlGn', 'Delta'),

    (232, GAMMA_S, 'Gamma Surface ×10⁴', 'plasma', 'Gamma'),

    (233, VANNA_S, 'Vanna Surface', 'coolwarm', 'Vanna'),

    (234, VOLGA_S, 'Volga Surface', 'viridis', 'Volga'),

    (235, PNL_S, 'PnL Surface ($)', 'RdYlGn', 'PnL')

]


for pos, Z, title, cmap, zlabel in surfaces:

    ax = fig3d.add_subplot(
        pos,
        projection='3d'
    )

    ax.set_facecolor('#0d0d0d')

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor('#222')
    ax.yaxis.pane.set_edgecolor('#222')
    ax.zaxis.pane.set_edgecolor('#222')

    ax.grid(True, color='#222')

    ax.tick_params(
        colors='#888',
        labelsize=6
    )

    surf = ax.plot_surface(
        S_g,
        IV_g*100,
        Z,
        cmap=cmap,
        alpha=0.9,
        linewidth=0
    )

    ax.set_title(
        title,
        color='white',
        fontsize=9
    )

    ax.set_xlabel(
        'Spot',
        fontsize=7
    )

    ax.set_ylabel(
        'IV %',
        fontsize=7
    )

    ax.set_zlabel(
        zlabel,
        fontsize=7
    )

    fig3d.colorbar(
        surf,
        ax=ax,
        shrink=0.40,
        aspect=12
    )


#══════════════════════════════════════
# GEX SURFACE
#══════════════════════════════════════

ax6d = fig3d.add_subplot(
    236,
    projection='3d'
)

ax6d.set_facecolor('#0d0d0d')

ax6d.xaxis.pane.fill = False
ax6d.yaxis.pane.fill = False
ax6d.zaxis.pane.fill = False

ax6d.xaxis.pane.set_edgecolor('#222')
ax6d.yaxis.pane.set_edgecolor('#222')
ax6d.zaxis.pane.set_edgecolor('#222')

ax6d.grid(True, color='#222')

ax6d.tick_params(colors='#888', labelsize=6)

T_range = np.linspace(

    5,

    60,

    40

)

K_range = np.linspace(

    GOLD_SPOT*0.88,

    GOLD_SPOT*1.12,

    50

)

KK, TT = np.meshgrid(

    K_range,

    T_range

)

GEX_3D = np.zeros_like(KK)


for i in range(KK.shape[0]):

    for j in range(KK.shape[1]):

        t_ = TT[i,j]/365

        m_ = KK[i,j]/GOLD_SPOT

        iv_ = ATM_IV + 0.03*(1-m_)*5

        iv_ = max(iv_,0.05)

        g_ = bs_greeks(

            GOLD_SPOT,

            KK[i,j],

            t_,

            RISK_FREE,

            iv_,

            'call'

        )

        GEX_3D[i,j] = (

            g_['gamma']

            *

            1000

            *

            GOLD_SPOT

        )


surf6 = ax6d.plot_surface(

    KK,

    TT,

    GEX_3D,

    cmap='plasma',

    linewidth=0,

    alpha=0.9

)

ax6d.set_title(

    'GEX Surface\n(Strike × DTE)',

    color='white',

    fontsize=9

)

ax6d.set_xlabel(

    'Strike',

    fontsize=7

)

ax6d.set_ylabel(

    'DTE',

    fontsize=7

)

ax6d.set_zlabel(

    'GEX',

    fontsize=7

)

fig3d.colorbar(

    surf6,

    ax=ax6d,

    shrink=0.40,

    aspect=12

)

plt.tight_layout()

plt.savefig(

    f'gold_greeks_3d_{today}.png',

    dpi=130,

    bbox_inches='tight',

    facecolor=BG

)

plt.show()

print()

print("✅ Figure 2 saved")

#══════════════════════════════════════════════
# FIGURE 3 — SPOTGAMMA FRAMEWORK
#══════════════════════════════════════════════

fig3, axes3 = plt.subplots(

    1,

    2,

    figsize=(18,7),

    facecolor=BG

)

fig3.suptitle(

    f'SpotGamma Framework — Gold\n{today}',

    color=GOLD,

    fontsize=13,

    fontweight='bold'

)


for ax in axes3:

    ax.set_facecolor('#111111')

    for sp in ax.spines.values():

        sp.set_color('#333')

    ax.tick_params(

        colors='#888',

        labelsize=8

    )


#══════════════════════════════════════
# LEFT PANEL
#══════════════════════════════════════

ax_l = axes3[0]

days_ = [0,1,2,3,4,5]

bull_path = [

    GOLD_SPOT*0.98,

    GOLD_SPOT*0.99,

    GOLD_SPOT*1.01,

    GOLD_SPOT*1.03,

    GOLD_SPOT*1.05,

    gamma_wall

]

bear_path = [

    GOLD_SPOT*0.98,

    GOLD_SPOT*0.97,

    GOLD_SPOT*0.95,

    GOLD_SPOT*0.93,

    gex_flip*1.01,

    gex_flip*0.98

]


ax_l.plot(

    days_,

    bull_path,

    color=GRN,

    lw=2.5,

    marker='o',

    ms=5,

    label='Bull Path'

)

ax_l.plot(

    days_,

    bear_path,

    color=RED,

    lw=2.5,

    marker='o',

    ms=5,

    label='Bear Path'

)


ax_l.axhline(

    GOLD_SPOT,

    color=GOLD,

    ls='--',

    lw=1,

    label=f'Spot ${GOLD_SPOT:,.0f}'

)

ax_l.axhline(

    gamma_wall,

    color=BLUE,

    ls=':',

    lw=1,

    label=f'Gamma Wall ${gamma_wall:,.0f}'

)

ax_l.axhline(

    call_wall,

    color=GRN,

    ls=':',

    lw=1,

    label=f'Call Wall ${call_wall:,.0f}'

)

ax_l.axhline(

    put_wall,

    color=RED,

    ls=':',

    lw=1,

    label=f'Put Wall ${put_wall:,.0f}'

)

ax_l.axhline(

    gex_flip,

    color='#FF8800',

    ls=':',

    lw=1,

    label=f'GEX Flip ${gex_flip:,.0f}'

)

ax_l.axhline(

    max_pain,

    color='#AAAAAA',

    ls=':',

    lw=1,

    label=f'Max Pain ${max_pain:,.0f}'

)

ax_l.set_xlabel(

    'Trading Days',

    color='#888'

)

ax_l.set_ylabel(

    'Gold Price',

    color='#888'

)

ax_l.set_title(

    'Bull vs Bear Paths',

    color='white'

)

ax_l.legend(

    fontsize=7,

    facecolor='#111',

    labelcolor='white'

)


#══════════════════════════════════════
# RIGHT PANEL
#══════════════════════════════════════

ax_r = axes3[1]

ax_r.axis('off')

if GOLD_SPOT > gex_flip:

    env = "LONG γ (pinning)"

    env_color = GRN

else:

    env = "SHORT γ (acceleration)"

    env_color = RED


rows = [

    ('Gamma Environment', env, env_color),

    ('','',None),

    ('Condition 1',

     'Call wall rising',

     GRN),

    ('Trade',

     f'Bull Call Spread\n{K_call:.0f}/{call_wall:.0f}',

     GRN),

    ('','',None),

    ('Condition 2',

     'Volatility collapsing',

     GOLD),

    ('Trade',

     f'Iron Condor\n{put_wall:.0f}-{call_wall:.0f}',

     GOLD),

    ('','',None),

    ('Condition 3',

     'Call wall stable',

     BLUE),

    ('Trade',

     'Calendar Spread',

     BLUE),

    ('','',None),

    ('Condition 4',

     'Break below GEX flip',

     RED),

    ('Trade',

     f'Put Spread\n{gex_flip:.0f}/{put_wall:.0f}',

     RED),

    ('','',None),

    ('Current Spot',

     f'${GOLD_SPOT:,.0f}',

     GOLD),

    ('Bias',

     'Bullish' if total_ev > 0 else 'Bearish',

     GRN if total_ev > 0 else RED)

]


y = 0.96

for label, value, color in rows:

    if label == '':

        y -= 0.03

        continue

    ax_r.text(

        0.02,

        y,

        label,

        transform=ax_r.transAxes,

        color='#888',

        fontsize=8

    )

    ax_r.text(

        0.48,

        y,

        value,

        transform=ax_r.transAxes,

        color=color,

        fontsize=8,

        fontweight='bold'

    )

    y -= 0.055


plt.tight_layout()

plt.savefig(

    f'gold_spotgamma_framework_{today}.png',

    dpi=130,

    bbox_inches='tight',

    facecolor=BG

)

plt.show()

print()

print("✅ Figure 3 saved")

#══════════════════════════════════════════════
# EXPECTED MOVE
#══════════════════════════════════════════════

expected_move = (

    GOLD_SPOT

    *

    ATM_IV

    *

    np.sqrt(T)

)

upper_move = GOLD_SPOT + expected_move
lower_move = GOLD_SPOT - expected_move

print()
print("="*80)
print("EXPECTED MOVE")
print("="*80)

print(f"1σ Move : ±${expected_move:,.0f}")

print(f"Upper Band : ${upper_move:,.0f}")

print(f"Lower Band : ${lower_move:,.0f}")


#══════════════════════════════════════════════
# DEALER POSITIONING SCORE
#══════════════════════════════════════════════

dealer_score = 0

if GOLD_SPOT > gex_flip:
    dealer_score += 1

if GOLD_SPOT > gamma_wall:
    dealer_score += 1

if call_wall > GOLD_SPOT:
    dealer_score += 1

if put_wall < GOLD_SPOT:
    dealer_score += 1

dealer_pct = dealer_score/4


if dealer_pct >= 0.75:

    dealer_state = "Strong Long Gamma"

    dealer_color = GRN

elif dealer_pct >= 0.50:

    dealer_state = "Moderate Long Gamma"

    dealer_color = GOLD

else:

    dealer_state = "Short Gamma"

    dealer_color = RED


print()
print("="*80)
print("DEALER POSITIONING")
print("="*80)

print("State:",dealer_state)

print("Score:",dealer_score,"/4")


#══════════════════════════════════════════════
# VOLATILITY TRIGGER
#══════════════════════════════════════════════

vol_trigger_up = call_wall

vol_trigger_down = gex_flip

print()
print("="*80)
print("VOLATILITY TRIGGERS")
print("="*80)

print(f"Upside Trigger   : ${vol_trigger_up:,.0f}")

print(f"Downside Trigger : ${vol_trigger_down:,.0f}")


#══════════════════════════════════════════════
# GAMMA ACCELERATION WARNING
#══════════════════════════════════════════════

distance_flip = abs(

    GOLD_SPOT - gex_flip

)/GOLD_SPOT


if distance_flip < 0.01:

    gamma_warning = "HIGH"

    gamma_color = RED

elif distance_flip < 0.03:

    gamma_warning = "MODERATE"

    gamma_color = GOLD

else:

    gamma_warning = "LOW"

    gamma_color = GRN


print()
print("="*80)
print("GAMMA ACCELERATION RISK")
print("="*80)

print(gamma_warning)


#══════════════════════════════════════════════
# TRADE ENGINE
#══════════════════════════════════════════════

if total_ev > 0:

    if GOLD_SPOT > gex_flip:

        trade = "Bull Call Spread"

        strike_buy = K_call

        strike_sell = call_wall

    else:

        trade = "Long Call"

        strike_buy = K_call

        strike_sell = None

else:

    if GOLD_SPOT < gex_flip:

        trade = "Bear Put Spread"

        strike_buy = gex_flip

        strike_sell = put_wall

    else:

        trade = "Iron Condor"

        strike_buy = put_wall

        strike_sell = call_wall


print()
print("="*80)
print("TRADE RECOMMENDATION")
print("="*80)

print("Strategy :",trade)

if strike_sell is not None:

    print(

        "Structure:",

        round(strike_buy),

        "/",

        round(strike_sell)

    )


#══════════════════════════════════════════════
# DAILY SUMMARY TABLE
#══════════════════════════════════════════════

summary = pd.DataFrame({

    'Metric':[

        'Spot',

        'ATM IV',

        'GEX Flip',

        'Gamma Wall',

        'Call Wall',

        'Put Wall',

        'Max Pain',

        'Expected Move',

        'Dealer State',

        'Gamma Risk',

        'Expected Value',

        'Trade'

    ],

    'Value':[

        f"${GOLD_SPOT:,.0f}",

        f"{ATM_IV*100:.1f}%",

        f"${gex_flip:,.0f}",

        f"${gamma_wall:,.0f}",

        f"${call_wall:,.0f}",

        f"${put_wall:,.0f}",

        f"${max_pain:,.0f}",

        f"±${expected_move:,.0f}",

        dealer_state,

        gamma_warning,

        f"${total_ev:+,.0f}",

        trade

    ]

})

print()
print("="*80)
print("DAILY SUMMARY")
print("="*80)

print(summary)


#══════════════════════════════════════════════
# SAVE SUMMARY CSV
#══════════════════════════════════════════════

summary.to_csv(

    f'gold_summary_{today}.csv',

    index=False

)

print()
print("✅ Summary saved")

print("✅ Figure 1 saved")

print("✅ Figure 2 saved")

print("✅ Figure 3 saved")

print()

print("🏁 DASHBOARD COMPLETE")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# ── CONFIG ────────────────────────────────────────────────────────────────────
GOLD_SPOT    = LIVE_GC   # was hardcoded 4593.0
ATM_IV       = 0.185        # ~18.5% implied vol (GVZ proxy)
RISK_FREE    = 0.053
DAYS_TO_EXP  = 30           # front-month focus
T            = DAYS_TO_EXP / 365
GC_MULT      = 100          # 1 GC contract = 100 oz
ACCOUNT      = 50_000
RISK_PCT     = 0.01

BG   = '#0d0d0d'
GOLD = '#FFD700'
RED  = '#FF4444'
GRN  = '#00FF88'
BLUE = '#4488FF'
MUTE = '#555555'

# ── BLACK-SCHOLES GREEKS ──────────────────────────────────────────────────────
def bs_greeks(S, K, T, r, sigma, flag='call'):
    if T <= 1e-6 or sigma <= 1e-6:
        intrinsic = max(S-K,0) if flag=='call' else max(K-S,0)
        return dict(price=intrinsic, delta=1.0 if S>K else 0.0,
                    gamma=0, vega=0, theta=0, vanna=0, charm=0, volga=0)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    phi  = norm.pdf(d1)
    Nd1  = norm.cdf(d1)  if flag=='call' else norm.cdf(-d1)
    Nd2  = norm.cdf(d2)  if flag=='call' else norm.cdf(-d2)
    sign = 1 if flag=='call' else -1
    price = sign*(S*Nd1 - K*np.exp(-r*T)*Nd2)
    delta = sign*Nd1
    gamma = phi / (S*sigma*np.sqrt(T))
    vega  = S*phi*np.sqrt(T)/100
    theta = (-(S*phi*sigma)/(2*np.sqrt(T))
             - sign*r*K*np.exp(-r*T)*(norm.cdf(sign*d2))) / 365
    vanna  = -phi*d2/sigma
    charm  = -phi*(2*r*T - d2*sigma*np.sqrt(T))/(2*T*sigma*np.sqrt(T))
    volga  = vega*(d1*d2/sigma)
    return dict(price=price, delta=delta, gamma=gamma, vega=vega,
                theta=theta, vanna=vanna, charm=charm, volga=volga)

# ── OPTION CHAIN — LIVE (was np.random.lognormal OI) ──────────────────────────
_live1 = fetch_gold_chain(max_dte=60, verbose=False)
strikes = np.sort(_live1.strike.unique())
expiries = sorted(_live1.expiry.unique().tolist())

chain = []
for _, _r in _live1.iterrows():
    exp = int(_r.expiry); t = exp/365
    K = float(_r.strike); m = K/GOLD_SPOT
    iv_c = float(_r.iv_call)
    iv_p = float(_r.iv_put)
    coi = int(_r.call_oi)
    poi = int(_r.put_oi)
    cg = bs_greeks(GOLD_SPOT, K, t, RISK_FREE, iv_c, 'call')
    pg = bs_greeks(GOLD_SPOT, K, t, RISK_FREE, iv_p, 'put')
    chain.append(dict(
        expiry=exp, strike=K, moneyness=m,
        iv_call=iv_c, iv_put=iv_p,
        call_oi=coi, put_oi=poi,
        call_price=cg['price'], put_price=pg['price'],
        call_delta=cg['delta'], put_delta=pg['delta'],
        call_gamma=cg['gamma'], put_gamma=pg['gamma'],
        call_vega=cg['vega'],  put_vega=pg['vega'],
        call_theta=cg['theta'],put_theta=pg['theta'],
        call_vanna=cg['vanna'],put_vanna=pg['vanna'],
        call_charm=cg['charm'],put_charm=pg['charm'],
        call_volga=cg['volga'],put_volga=pg['volga'],
    ))

df = pd.DataFrame(chain)

# Net GEX / Vanna / Charm per strike (aggregate all expiries)
by_K = df.groupby('strike').apply(lambda x: pd.Series({
    'net_gex'  : ((x.call_gamma*x.call_oi - x.put_gamma*x.put_oi)*100*GOLD_SPOT).sum(),
    'net_vanna': ((x.call_vanna*x.call_oi - x.put_vanna*x.put_oi)*100).sum(),
    'net_charm': ((x.call_charm*x.call_oi - x.put_charm*x.put_oi)*100).sum(),
    'net_vega' : ((x.call_vega*x.call_oi  + x.put_vega*x.put_oi)*100).sum(),
    'put_oi'   : x.put_oi.sum(),
    'call_oi'  : x.call_oi.sum(),
})).reset_index()

# Key levels
sorted_K   = by_K.sort_values('strike')
gex_flip, _c1 = true_gex_flip(_live1, GOLD_SPOT)
gex_flip = gex_flip if gex_flip is not None else GOLD_SPOT*0.92
gamma_wall = float(by_K.loc[by_K.net_gex.abs().idxmax(),'strike'])
call_wall, put_wall = side_walls(by_K, GOLD_SPOT)   # side-restricted

max_pain = true_max_pain(_live1, max_dte=14)   # proper max pain
# near_K.columns=['strike','total_oi']   # orphaned: near_K no longer built (max_pain via true_max_pain)
# max_pain already computed above by true_max_pain() — old idxmin line removed

print(f"Gold Spot      : ${GOLD_SPOT:,.2f}")
print(f"GEX Flip       : ${gex_flip:,.2f}")
print(f"Gamma Wall     : ${gamma_wall:,.2f}")
print(f"Call Wall      : ${call_wall:,.2f}")
print(f"Put Wall       : ${put_wall:,.2f}")
print(f"Max Pain (14d) : ${max_pain:,.2f}")

# ── TESLA-STYLE PNL TABLE FOR GOLD CALL ──────────────────────────────────────
print("\n" + "="*65)
print(" GOLD EXPECTED VALUE TABLE — BUY ATM CALL (30d)")
print("="*65)
K_call  = round(GOLD_SPOT/10)*10   # ATM strike
g0      = bs_greeks(GOLD_SPOT, K_call, T, RISK_FREE, ATM_IV, 'call')
premium = g0['price']
scenarios = [-0.10, -0.05, 0.0, +0.05, +0.10, +0.15]
probs     = [0.10,   0.15,  0.30, 0.25, 0.12,  0.08]   # your view

# Market-implied probabilities from ATM IV, for comparison. If EV is only
# positive under `probs` and not under these, the edge is your VIEW, not the
# structure — size it accordingly.
from scipy.stats import norm as _n
_T = DAYS_TO_EXP / 365
_v = (ATM_IV if 'ATM_IV' in dir() else 0.24) * np.sqrt(_T)
_mu = -0.5 * ((ATM_IV if 'ATM_IV' in dir() else 0.24) ** 2) * _T
_lv = np.array([GOLD_SPOT * (1 + m) for m in scenarios])
_ed = np.concatenate([[-np.inf], (_lv[:-1] + _lv[1:]) / 2, [np.inf]])
probs_mkt = [
    _n.cdf(np.inf if _ed[i+1] == np.inf else (np.log(_ed[i+1]/GOLD_SPOT) - _mu)/_v)
  - _n.cdf(-np.inf if _ed[i] == -np.inf else (np.log(_ed[i]/GOLD_SPOT) - _mu)/_v)
    for i in range(len(_lv))
]

print(f"\n  Strike: ${K_call:,.0f} | Premium paid: ${premium:.2f}/oz | "
      f"Contract cost: ${premium*GC_MULT:,.0f}")
print(f"\n  {'Scenario':>10} {'Spot':>8} {'Delta+Γ PnL':>12} "
      f"{'Theta':>8} {'Vega':>8} {'Total PnL':>10} {'Prob':>7} {'EV':>9}")
print("  " + "-"*80)

total_ev = 0
for chg, prob in zip(scenarios, probs):
    S_new   = GOLD_SPOT*(1+chg)
    days_held = DAYS_TO_EXP
    t_new   = 1/365  # near expiry
    iv_new  = ATM_IV*(1 - 0.3*chg)  # IV crush on rally, spike on drop
    g_new   = bs_greeks(S_new, K_call, t_new, RISK_FREE, max(iv_new,0.05), 'call')
    total_pnl = (g_new['price'] - premium) * GC_MULT
    # decompose
    dg_pnl = (g0['delta']*(S_new-GOLD_SPOT) +
               0.5*g0['gamma']*(S_new-GOLD_SPOT)**2) * GC_MULT
    th_pnl = g0['theta'] * days_held * GC_MULT
    vg_pnl = g0['vega']  * (iv_new - ATM_IV)*100 * GC_MULT
    ev      = prob * total_pnl
    total_ev += ev
    tag = f"{chg:+.0%}"
    print(f"  {tag:>10} {S_new:>8,.0f} {dg_pnl:>+12.0f} "
          f"{th_pnl:>+8.0f} {vg_pnl:>+8.0f} {total_pnl:>+10.0f} "
          f"{prob:>6.0%} {ev:>+9.0f}")

print("  " + "-"*80)
print(f"  {'EXPECTED VALUE':>55}  ${total_ev:>+9.0f}")
verdict = "POSITIVE EV ✅" if total_ev > 0 else "NEGATIVE EV ❌"
print(f"  {verdict}")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — GREEKS DASHBOARD (2D)
# ══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20,14), facecolor=BG)
fig.suptitle(f'GOLD OPTIONS — Greeks & GEX Dashboard | Spot ${GOLD_SPOT:,.0f}',
             color=GOLD, fontsize=14, fontweight='bold', y=0.98)

gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
ax1 = fig.add_subplot(gs[0,0])   # Net GEX
ax2 = fig.add_subplot(gs[0,1])   # Net Vanna
ax3 = fig.add_subplot(gs[0,2])   # Net Charm
ax4 = fig.add_subplot(gs[1,0])   # Vol smile (30d)
ax5 = fig.add_subplot(gs[1,1])   # Delta vs Strike
ax6 = fig.add_subplot(gs[1,2])   # Gamma vs Strike
ax7 = fig.add_subplot(gs[2,0])   # Call/Put OI
ax8 = fig.add_subplot(gs[2,1])   # PnL scenarios
ax9 = fig.add_subplot(gs[2,2])   # Scorecard

for ax in [ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9]:
    ax.set_facecolor('#111111')
    for sp in ax.spines.values(): sp.set_color('#333')
    ax.tick_params(colors='#888', labelsize=8)

def vline(ax, x, color, label):
    ax.axvline(x, color=color, lw=0.9, ls='--', alpha=0.8)
    ax.text(x, ax.get_ylim()[1]*0.92, f' {label}\n${x:,.0f}',
            color=color, fontsize=6.5, va='top')

ks  = by_K.strike.values
gex = by_K.net_gex.values
van = by_K.net_vanna.values
cha = by_K.net_charm.values

# — GEX —
ax1.bar(ks, gex, width=15, color=[GRN if v>=0 else RED for v in gex], alpha=0.85)
ax1.axhline(0, color=MUTE, lw=0.5)
ax1.axvline(GOLD_SPOT, color=GOLD, lw=1.2, label=f'Spot ${GOLD_SPOT:,.0f}')
ax1.axvline(gamma_wall, color=BLUE, lw=0.8, ls='--', label=f'γ Wall ${gamma_wall:,.0f}')
ax1.axvline(gex_flip, color=RED, lw=0.8, ls='--', label=f'GEX Flip ${gex_flip:,.0f}')
ax1.set_title('Net GEX by Strike', color='white', fontsize=9)
ax1.set_ylabel('Net GEX ($)', color='#888', fontsize=8)
ax1.legend(fontsize=6, facecolor='#111', labelcolor='white')
ax1.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# — Vanna —
ax2.bar(ks, van, width=15, color=['#00CCBB' if v>=0 else '#FFA500' for v in van], alpha=0.85)
ax2.axhline(0, color=MUTE, lw=0.5)
ax2.axvline(GOLD_SPOT, color=GOLD, lw=1.2)
ax2.set_title('Net Vanna by Strike\n(IV↑ → dealer delta hedge)', color='white', fontsize=9)
ax2.set_ylabel('Net Vanna', color='#888', fontsize=8)
ax2.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# — Charm —
ax3.bar(ks, cha, width=15, color=['#AA66FF' if v>=0 else '#888888' for v in cha], alpha=0.85)
ax3.axhline(0, color=MUTE, lw=0.5)
ax3.axvline(GOLD_SPOT, color=GOLD, lw=1.2)
ax3.set_title('Net Charm by Strike\n(delta decay toward expiry)', color='white', fontsize=9)
ax3.set_ylabel('Net Charm', color='#888', fontsize=8)
ax3.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# — Vol Smile —
smile_df = df[df.expiry==30].copy()
ax4.plot(smile_df.strike, smile_df.iv_call*100, color=GRN,  lw=1.5, label='Call IV')
ax4.plot(smile_df.strike, smile_df.iv_put*100,  color=RED,  lw=1.5, label='Put IV')
ax4.axvline(GOLD_SPOT, color=GOLD, lw=1.0, ls='--')
ax4.set_title('Vol Smile — 30d', color='white', fontsize=9)
ax4.set_ylabel('IV %', color='#888', fontsize=8)
ax4.legend(fontsize=7, facecolor='#111', labelcolor='white')

# — Delta —
d30 = df[df.expiry==30]
ax5.plot(d30.strike, d30.call_delta, color=GRN, lw=1.5, label='Call Δ')
ax5.plot(d30.strike, d30.put_delta,  color=RED, lw=1.5, label='Put Δ')
ax5.axvline(GOLD_SPOT, color=GOLD, lw=1.0, ls='--')
ax5.axhline(0, color=MUTE, lw=0.4)
ax5.set_title('Delta vs Strike — 30d', color='white', fontsize=9)
ax5.set_ylabel('Delta', color='#888', fontsize=8)
ax5.legend(fontsize=7, facecolor='#111', labelcolor='white')

# — Gamma —
ax6.plot(d30.strike, d30.call_gamma*1e4, color=BLUE, lw=1.5, label='Call γ ×10⁴')
ax6.axvline(GOLD_SPOT, color=GOLD, lw=1.0, ls='--')
ax6.set_title('Gamma vs Strike — 30d', color='white', fontsize=9)
ax6.set_ylabel('Gamma ×10⁴', color='#888', fontsize=8)
ax6.legend(fontsize=7, facecolor='#111', labelcolor='white')

# — OI bar —
ax7.bar(by_K.strike, by_K.call_oi/1000, width=12, color=GRN,  alpha=0.7, label='Call OI')
ax7.bar(by_K.strike, -by_K.put_oi/1000, width=12, color=RED,  alpha=0.7, label='Put OI')
ax7.axvline(GOLD_SPOT, color=GOLD, lw=1.2)
ax7.axvline(call_wall, color=GRN, lw=0.8, ls='--', label=f'Call Wall ${call_wall:,.0f}')
ax7.axvline(put_wall,  color=RED, lw=0.8, ls='--', label=f'Put Wall  ${put_wall:,.0f}')
ax7.axhline(0, color=MUTE, lw=0.4)
ax7.set_title('Call / Put Open Interest (k)', color='white', fontsize=9)
ax7.set_ylabel('OI (thousands)', color='#888', fontsize=8)
ax7.legend(fontsize=6, facecolor='#111', labelcolor='white')

# — PnL scenario bar —
scen_labels = [f'{c:+.0%}' for c in scenarios]
pnls = []
for chg in scenarios:
    S_new = GOLD_SPOT*(1+chg)
    g_new = bs_greeks(S_new, K_call, 1/365, RISK_FREE,
                      max(ATM_IV*(1-0.3*chg),0.05), 'call')
    pnls.append((g_new['price']-premium)*GC_MULT)

colors_bar = [GRN if p>=0 else RED for p in pnls]
bars = ax8.bar(scen_labels, pnls, color=colors_bar, alpha=0.85)
ax8.axhline(0, color=MUTE, lw=0.5)
for bar,val in zip(bars,pnls):
    ax8.text(bar.get_x()+bar.get_width()/2,
             val+(50 if val>=0 else -150),
             f'${val:+,.0f}', ha='center', fontsize=7.5, color='white')
ax8.set_title(f'ATM Call PnL at Expiry\nStrike ${K_call:,.0f} | Cost ${premium:.1f}/oz',
              color='white', fontsize=9)
ax8.set_ylabel('PnL per contract ($)', color='#888', fontsize=8)

# — Scorecard —
ax9.axis('off')
ax9.set_title('GEX Scorecard', color=GOLD, fontsize=10)
items = [
    ('Spot',           f'${GOLD_SPOT:,.0f}',                       'white'),
    ('GEX Flip',       f'${gex_flip:,.0f}',                        RED),
    ('Gamma Wall',     f'${gamma_wall:,.0f}',                       BLUE),
    ('Call Wall',      f'${call_wall:,.0f}',                        GRN),
    ('Put Wall',       f'${put_wall:,.0f}',                        RED),
    ('Max Pain (14d)', f'${max_pain:,.0f}',                        GOLD),
    ('ATM IV',         f'{ATM_IV*100:.1f}%',                       'white'),
    ('ATM Premium',    f'${premium:.2f}/oz  (${premium*GC_MULT:,.0f}/contract)', 'white'),
    ('Env (vs flip)',  'LONG γ — pins price' if GOLD_SPOT>gex_flip else 'SHORT γ — amplifies',
                       GRN if GOLD_SPOT>gex_flip else RED),
    ('EV (ATM call)',  f'${total_ev:+,.0f}',
                       GRN if total_ev>0 else RED),
]
y = 0.95
for label, val, col in items:
    ax9.text(0.02, y, label, transform=ax9.transAxes,
             color='#888', fontsize=8.5)
    ax9.text(0.98, y, val, transform=ax9.transAxes,
             color=col, fontsize=8.5, ha='right', fontweight='bold')
    y -= 0.092

plt.savefig('gold_greeks_2d.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Figure 1 saved → gold_greeks_2d.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — 3D SURFACES
# ══════════════════════════════════════════════════════════════════════════════
from mpl_toolkits.mplot3d import Axes3D

spot_range = np.linspace(GOLD_SPOT*0.85, GOLD_SPOT*1.15, 60)
iv_range   = np.linspace(0.10, 0.30, 50)
S_g, IV_g  = np.meshgrid(spot_range, iv_range)

K_atm = K_call
t30   = 30/365

DELTA_S = np.zeros_like(S_g)
GAMMA_S = np.zeros_like(S_g)
VANNA_S = np.zeros_like(S_g)
VOLGA_S = np.zeros_like(S_g)
PNL_S   = np.zeros_like(S_g)

for i in range(S_g.shape[0]):
    for j in range(S_g.shape[1]):
        g = bs_greeks(S_g[i,j], K_atm, t30, RISK_FREE, IV_g[i,j], 'call')
        DELTA_S[i,j] = g['delta']
        GAMMA_S[i,j] = g['gamma']*1e4
        VANNA_S[i,j] = g['vanna']
        VOLGA_S[i,j] = g['volga']
        PNL_S[i,j]   = (g['price'] - premium)*GC_MULT

fig3d = plt.figure(figsize=(22,16), facecolor=BG)
fig3d.suptitle(f'GOLD OPTIONS — 3D Greek Surfaces | ATM Strike ${K_atm:,.0f}',
               color=GOLD, fontsize=14, fontweight='bold')

surfaces = [
    (231, DELTA_S, 'Delta Surface',        'RdYlGn',  'Delta'),
    (232, GAMMA_S, 'Gamma Surface ×10⁴',   'plasma',  'Gamma ×10⁴'),
    (233, VANNA_S, 'Vanna Surface\n(dΔ/dσ)', 'coolwarm','Vanna'),
    (234, VOLGA_S, 'Volga Surface\n(dVega/dσ)','viridis', 'Volga'),
    (235, PNL_S,   'PnL Surface\n($/contract)','RdYlGn', 'PnL $'),
]

for pos, Z, title, cmap, zlabel in surfaces:
    ax = fig3d.add_subplot(pos, projection='3d')
    ax.set_facecolor('#0a0a0a')
    surf = ax.plot_surface(S_g, IV_g*100, Z,
                           cmap=cmap, alpha=0.88,
                           linewidth=0, antialiased=True)
    ax.set_xlabel('Spot $', color='#888', fontsize=7, labelpad=4)
    ax.set_ylabel('IV %',   color='#888', fontsize=7, labelpad=4)
    ax.set_zlabel(zlabel,   color='#888', fontsize=7, labelpad=4)
    ax.set_title(title, color='white', fontsize=9, pad=8)
    ax.tick_params(colors='#666', labelsize=6)
    ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('#222')
    ax.yaxis.pane.set_edgecolor('#222')
    ax.zaxis.pane.set_edgecolor('#222')
    ax.grid(True, color='#1a1a1a', lw=0.4)
    # Mark current spot
    ax.axvline(GOLD_SPOT, color=GOLD, lw=0.8, alpha=0.6)
    fig3d.colorbar(surf, ax=ax, shrink=0.4, pad=0.08,
                   aspect=12).ax.tick_params(colors='#888', labelsize=6)

# 6th panel — GEX heatmap over time
ax6d = fig3d.add_subplot(236, projection='3d')
ax6d.set_facecolor('#0a0a0a')
T_range = np.linspace(5, 60, 40)
K_range = np.linspace(GOLD_SPOT*0.88, GOLD_SPOT*1.12, 50)
KK, TT  = np.meshgrid(K_range, T_range)
GEX_3D  = np.zeros_like(KK)
for i in range(KK.shape[0]):
    for j in range(KK.shape[1]):
        t_  = TT[i,j]/365
        m_  = KK[i,j]/GOLD_SPOT
        iv_ = ATM_IV + 0.03*(1-m_)*5
        g_  = bs_greeks(GOLD_SPOT, KK[i,j], t_, RISK_FREE, max(iv_,0.05))
        GEX_3D[i,j] = g_['gamma']*1000*GOLD_SPOT

surf6 = ax6d.plot_surface(KK, TT, GEX_3D, cmap='plasma',
                          alpha=0.88, linewidth=0, antialiased=True)
ax6d.set_xlabel('Strike $', color='#888', fontsize=7, labelpad=4)
ax6d.set_ylabel('DTE',      color='#888', fontsize=7, labelpad=4)
ax6d.set_zlabel('GEX',      color='#888', fontsize=7, labelpad=4)
ax6d.set_title('GEX Surface\n(Strike × DTE)', color='white', fontsize=9, pad=8)
ax6d.tick_params(colors='#666', labelsize=6)
for pane in [ax6d.xaxis.pane, ax6d.yaxis.pane, ax6d.zaxis.pane]:
    pane.fill = False; pane.set_edgecolor('#222')
ax6d.grid(True, color='#1a1a1a', lw=0.4)
fig3d.colorbar(surf6, ax=ax6d, shrink=0.4, pad=0.08,
               aspect=12).ax.tick_params(colors='#888', labelsize=6)

plt.tight_layout()
plt.savefig('gold_greeks_3d.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Figure 2 saved → gold_greeks_3d.png")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — SPOTGAMMA TRADE FRAMEWORK APPLIED TO GOLD
# ══════════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)
fig3.suptitle('SpotGamma Framework — Gold Trade Scenarios', color=GOLD,
              fontsize=13, fontweight='bold')

for ax in axes3:
    ax.set_facecolor('#111111')
    for sp in ax.spines.values(): sp.set_color('#333')
    ax.tick_params(colors='#888', labelsize=8)

# Left: Scenario path diagram
ax_l = axes3[0]
price_path_bull = [GOLD_SPOT*0.98, GOLD_SPOT*0.99, GOLD_SPOT*1.01,
                   GOLD_SPOT*1.03, GOLD_SPOT*1.06, gamma_wall]
price_path_bear = [GOLD_SPOT*0.98, GOLD_SPOT*0.97, GOLD_SPOT*0.95,
                   GOLD_SPOT*0.93, gex_flip*1.01, gex_flip*0.98]
days_ = [0,1,2,3,4,5]

ax_l.plot(days_, price_path_bull, color=GRN,  lw=2.2, label='Bull path → Gamma Wall', marker='o', ms=5)
ax_l.plot(days_, price_path_bear, color=RED,   lw=2.2, label='Bear path → GEX Flip', marker='o', ms=5)
ax_l.axhline(GOLD_SPOT,  color=GOLD, lw=1.0, ls='--', label=f'Spot ${GOLD_SPOT:,.0f}')
ax_l.axhline(gamma_wall, color=BLUE, lw=0.9, ls=':', label=f'Gamma Wall ${gamma_wall:,.0f}')
ax_l.axhline(call_wall,  color=GRN,  lw=0.7, ls=':', label=f'Call Wall ${call_wall:,.0f}')
ax_l.axhline(put_wall,   color=RED,  lw=0.7, ls=':', label=f'Put Wall ${put_wall:,.0f}')
ax_l.axhline(gex_flip,   color='#FF8800', lw=0.9, ls=':', label=f'GEX Flip ${gex_flip:,.0f}')
ax_l.axhline(max_pain,   color='#888888', lw=0.7, ls=':', label=f'Max Pain ${max_pain:,.0f}')
ax_l.set_xlabel('Trading days', color='#888', fontsize=9)
ax_l.set_ylabel('Gold Price $', color='#888', fontsize=9)
ax_l.set_title('Bull vs Bear Path — SpotGamma Levels', color='white', fontsize=10)
ax_l.legend(fontsize=7.5, facecolor='#111', labelcolor='white', loc='center left')
ax_l.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Right: trade table
ax_r = axes3[1]
ax_r.axis('off')
ax_r.set_title('SpotGamma Trade Framework — Gold', color=GOLD, fontsize=10)

env = 'LONG γ (pins)' if GOLD_SPOT > gex_flip else 'SHORT γ (amplifies)'
env_col = GRN if GOLD_SPOT > gex_flip else RED
rows = [
    ('GAMMA ENV',     env,                                              env_col),
    ('','',''),
    ('CONDITION 1', 'Rally + Call Wall shifts higher',                  GRN),
    ('→ POV',       'Bullish',                                          GRN),
    ('→ TRADE',     f'Call spread: BUY ${K_atm:,.0f} / SELL ${call_wall:,.0f}', GRN),
    ('','',''),
    ('CONDITION 2', 'RVol collapsing near Call Wall',                   GOLD),
    ('→ POV',       'Street choking on gamma',                          GOLD),
    ('→ TRADE',     f'Iron Condor expiry: ${put_wall:,.0f}–${call_wall:,.0f}',  GOLD),
    ('','',''),
    ('CONDITION 3', 'Call Wall stable + low RVol',                      BLUE),
    ('→ POV',       'Moderately bullish',                               BLUE),
    ('→ TRADE',     'Calendar spread: SELL front / BUY back',           BLUE),
    ('','',''),
    ('CONDITION 4', 'Vol Trigger break to downside',                    RED),
    ('→ POV',       'Acceleration lower',                               RED),
    ('→ TRADE',     f'Put spread: BUY ${gex_flip:,.0f}p / SELL ${put_wall:,.0f}p', RED),
    ('','',''),
    ('CURRENT SPOT', f'${GOLD_SPOT:,.0f} — above GEX flip',             GOLD),
    ('BIAS',         'Long gamma environment → pinning tendency',        GRN),
]

y = 0.97
for label, val, col in rows:
    if label == '':
        y -= 0.025
        continue
    bold = label in ('GAMMA ENV','CURRENT SPOT','BIAS')
    ax_r.text(0.01, y, label, transform=ax_r.transAxes,
              color='#777' if not bold else GOLD,
              fontsize=8 if not bold else 9,
              fontweight='bold' if bold else 'normal')
    ax_r.text(0.40, y, val, transform=ax_r.transAxes,
              color=col, fontsize=8,
              fontweight='bold' if bold else 'normal')
    y -= 0.048

plt.tight_layout()
plt.savefig('gold_spotgamma_framework.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Figure 3 saved → gold_spotgamma_framework.png")
print("\n🏁 All done. Three figures generated.")